**INSTALLATION OF THE DEPENDENCES**

In [ ]:
!sudo pip install torch transformers numpy pandas tqdm requests

**Baseline Benchmark (1)**

In [7]:
import json
import requests
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Load dataset
# -------------------------
def load_dataset():
    url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json"
    data = requests.get(url).json()

    dataset = []
    for item in data:
        mc1 = item["mc1_targets"]
        choices = list(mc1.keys())
        labels = list(mc1.values())

        if len(choices) < 2:
            continue

        correct = labels.index(1)

        dataset.append({
            "question": item["question"],
            "choices": choices,
            "label": correct
        })

    return dataset

# -------------------------
# Prompt builder
# -------------------------
def build_prompt(question, choices):
    return f"Question: {question}\nAnswer:"
    return f"""Question: {question}
Options:
{options}

Answer with the correct option text only.
"""

# -------------------------
# Score choices
# -------------------------
@torch.no_grad()
def score_choices(model, tokenizer, prompt, choices):
    inputs = tokenizer(
        [prompt] * len(choices),
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    targets = tokenizer(
        choices,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    # Mask padding tokens properly
    labels = targets.input_ids.clone()
    labels[targets.attention_mask == 0] = -100

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )

    logits = outputs.logits
    log_probs = F.log_softmax(logits, dim=-1)

    target_ids = targets.input_ids
    mask = targets.attention_mask.bool()

    # shift for decoder alignment
    log_probs = log_probs[:, :-1, :]
    target_ids = target_ids[:, 1:]
    mask = mask[:, 1:]

    gathered = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    gathered[~mask] = 0

    seq_logprob = gathered.sum(dim=1) / mask.sum(dim=1)

    return seq_logprob.cpu().numpy()
# -------------------------
# Main
# -------------------------
def main():
    print("Loading dataset...")
    dataset = load_dataset()
    print(f"Total examples: {len(dataset)}")

    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    correct = 0

    print("Running baseline evaluation...")
    for ex in tqdm(dataset):
        prompt = build_prompt(ex["question"], ex["choices"])
        scores = score_choices(model, tokenizer, prompt, ex["choices"])

        pred = int(np.argmax(scores))

        if pred == ex["label"]:
            correct += 1

    acc = correct / len(dataset)

    print("\n=== BASELINE RESULTS ===")
    print(f"Accuracy: {acc:.4f}")


if __name__ == "__main__":
    main()

Loading dataset...
Total examples: 790
Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running baseline evaluation...


100%|██████████| 790/790 [00:34<00:00, 23.12it/s]


=== BASELINE RESULTS ===
Accuracy: 0.2557


**BENCHMARKING WIRH ENTROPY AS A FUNCTION**

In [11]:
import json
import requests
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Load dataset
# -------------------------
def load_dataset():
    url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json"
    data = requests.get(url).json()

    dataset = []
    for item in data:
        mc1 = item["mc1_targets"]
        choices = list(mc1.keys())
        labels = list(mc1.values())

        if len(choices) < 2:
            continue

        correct = labels.index(1)

        dataset.append({
            "question": item["question"],
            "choices": choices,
            "label": correct
        })

    return dataset

# -------------------------
# Prompt (neutral)
# -------------------------
def build_prompt(question):
    return f"Question: {question}\nAnswer:"

# -------------------------
# Score choices
# -------------------------
@torch.no_grad()
def score_choices(model, tokenizer, prompt, choices):
    batch_size = len(choices)

    inputs = tokenizer(
        [prompt] * batch_size,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    targets = tokenizer(
        choices,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    labels = targets.input_ids.clone()
    labels[targets.attention_mask == 0] = -100

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )

    logits = outputs.logits
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()

    target_ids = targets.input_ids
    mask = targets.attention_mask.bool()

    # shift
    log_probs = log_probs[:, :-1, :]
    probs = probs[:, :-1, :]
    target_ids = target_ids[:, 1:]
    mask = mask[:, 1:]

    # lengths (safe)
    lengths = mask.sum(dim=1).clamp(min=1)

    # logprob
    gathered = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    gathered = gathered * mask
    seq_logprob = gathered.sum(dim=1) / lengths

    # entropy
    token_entropy = -(probs * log_probs).sum(dim=-1)
    token_entropy = token_entropy * mask
    seq_entropy = token_entropy.sum(dim=1) / lengths

    # convert safely
    seq_logprob = seq_logprob.detach().cpu().numpy()
    seq_entropy = seq_entropy.detach().cpu().numpy()

    return seq_logprob, seq_entropy

# -------------------------
# Main
# -------------------------
def main():
    print("Loading dataset...")
    dataset = load_dataset()
    print(f"Total examples: {len(dataset)}")

    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # safety: ensure pad token exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    correct = 0
    correct_entropy = []
    wrong_entropy = []

    print("Running baseline evaluation...")

    for ex in tqdm(dataset):
        prompt = build_prompt(ex["question"])

        scores, entropies = score_choices(
            model, tokenizer, prompt, ex["choices"]
        )

        pred = int(np.argmax(scores))

        ent = entropies[pred]

        if pred == ex["label"]:
            correct += 1
            if not np.isnan(ent):
                correct_entropy.append(ent)
        else:
            if not np.isnan(ent):
                wrong_entropy.append(ent)

    acc = correct / len(dataset)

    print("\n=== BASELINE RESULTS ===")
    print(f"Accuracy: {acc:.4f}")

    print("\n=== ENTROPY ANALYSIS ===")

    if len(correct_entropy) > 0:
        print(f"Correct mean entropy: {np.mean(correct_entropy):.4f}")
    else:
        print("Correct mean entropy: N/A")

    if len(wrong_entropy) > 0:
        print(f"Wrong mean entropy:   {np.mean(wrong_entropy):.4f}")
    else:
        print("Wrong mean entropy:   N/A")

# -------------------------
if __name__ == "__main__":
    main()

Loading dataset...
Total examples: 790
Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running baseline evaluation...


100%|██████████| 790/790 [00:34<00:00, 23.03it/s]


=== BASELINE RESULTS ===
Accuracy: 0.2557

=== ENTROPY ANALYSIS ===
Correct mean entropy: 10.3649
Wrong mean entropy:   10.0650


**CORRELATION BETWEEN ENTROPY AND CORRECTNESS**

In [12]:
import json
import requests
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from scipy.stats import pearsonr

# -------------------------
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
def load_dataset():
    url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json"
    data = requests.get(url).json()

    dataset = []
    for item in data:
        mc1 = item["mc1_targets"]
        choices = list(mc1.keys())
        labels = list(mc1.values())

        if len(choices) < 2:
            continue

        correct = labels.index(1)

        dataset.append({
            "question": item["question"],
            "choices": choices,
            "label": correct
        })

    return dataset

# -------------------------
def build_prompt(question):
    return f"Question: {question}\nAnswer:"

# -------------------------
@torch.no_grad()
def score_choices(model, tokenizer, prompt, choices):
    inputs = tokenizer(
        [prompt] * len(choices),
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    targets = tokenizer(
        choices,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    labels = targets.input_ids.clone()
    labels[targets.attention_mask == 0] = -100

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )

    logits = outputs.logits
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()

    target_ids = targets.input_ids
    mask = targets.attention_mask.bool()

    # shift
    log_probs = log_probs[:, :-1, :]
    probs = probs[:, :-1, :]
    target_ids = target_ids[:, 1:]
    mask = mask[:, 1:]

    lengths = mask.sum(dim=1).clamp(min=1)

    # logprob
    gathered = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    gathered = gathered * mask
    seq_logprob = gathered.sum(dim=1) / lengths

    # entropy
    token_entropy = -(probs * log_probs).sum(dim=-1)
    token_entropy = token_entropy * mask
    seq_entropy = token_entropy.sum(dim=1) / lengths

    return seq_logprob.cpu().numpy(), seq_entropy.cpu().numpy()

# -------------------------
def main():
    print("Loading dataset...")
    dataset = load_dataset()
    print(f"Total examples: {len(dataset)}")

    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    correct = 0
    correct_entropy = []
    wrong_entropy = []

    # analysis tracking
    all_correct_flags = []
    all_entropies = []

    entropy_choice_correct = 0
    oracle_correct = 0

    print("Running evaluation...")

    for ex in tqdm(dataset):
        prompt = build_prompt(ex["question"])

        scores, entropies = score_choices(
            model, tokenizer, prompt, ex["choices"]
        )

        pred = int(np.argmax(scores))
        entropy_pred = int(np.argmin(entropies))

        gold = ex["label"]

        # baseline accuracy
        if pred == gold:
            correct += 1
            correct_entropy.append(entropies[pred])
            all_correct_flags.append(1)
        else:
            wrong_entropy.append(entropies[pred])
            all_correct_flags.append(0)

        all_entropies.append(entropies[pred])

        # entropy ranking test
        if entropy_pred == gold:
            entropy_choice_correct += 1

        # oracle (best of both)
        if pred == gold or entropy_pred == gold:
            oracle_correct += 1

    acc = correct / len(dataset)
    entropy_acc = entropy_choice_correct / len(dataset)
    oracle_acc = oracle_correct / len(dataset)

    print("\n=== BASELINE RESULTS ===")
    print(f"Accuracy: {acc:.4f}")

    print("\n=== ENTROPY ANALYSIS ===")
    print(f"Correct mean entropy: {np.mean(correct_entropy):.4f}")
    print(f"Wrong mean entropy:   {np.mean(wrong_entropy):.4f}")

    # correlation
    corr, _ = pearsonr(all_entropies, all_correct_flags)
    print(f"\nEntropy–Correctness Correlation: {corr:.4f}")

    print("\n=== RANKING TEST ===")
    print(f"Entropy-only accuracy: {entropy_acc:.4f}")

    print("\n=== ORACLE TEST ===")
    print(f"Oracle (best of both): {oracle_acc:.4f}")

# -------------------------
if __name__ == "__main__":
    main()

Loading dataset...
Total examples: 790
Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running evaluation...


100%|██████████| 790/790 [00:34<00:00, 22.66it/s]


=== BASELINE RESULTS ===
Accuracy: 0.2557

=== ENTROPY ANALYSIS ===
Correct mean entropy: 10.3649
Wrong mean entropy:   10.0650

Entropy–Correctness Correlation: 0.0870

=== RANKING TEST ===
Entropy-only accuracy: 0.2342

=== ORACLE TEST ===
Oracle (best of both): 0.4076


**CHASING THE ORACLE Method 1**

In [13]:
import json
import requests
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from scipy.stats import pearsonr

# -------------------------
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
def load_dataset():
    url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json"
    data = requests.get(url).json()

    dataset = []
    for item in data:
        mc1 = item["mc1_targets"]
        choices = list(mc1.keys())
        labels = list(mc1.values())

        if len(choices) < 2:
            continue

        correct = labels.index(1)

        dataset.append({
            "question": item["question"],
            "choices": choices,
            "label": correct
        })

    return dataset

# -------------------------
def build_prompt(question):
    return f"Question: {question}\nAnswer:"

# -------------------------
@torch.no_grad()
def score_choices(model, tokenizer, prompt, choices):
    inputs = tokenizer(
        [prompt] * len(choices),
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    targets = tokenizer(
        choices,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    labels = targets.input_ids.clone()
    labels[targets.attention_mask == 0] = -100

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )

    logits = outputs.logits
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()

    target_ids = targets.input_ids
    mask = targets.attention_mask.bool()

    # shift
    log_probs = log_probs[:, :-1, :]
    probs = probs[:, :-1, :]
    target_ids = target_ids[:, 1:]
    mask = mask[:, 1:]

    lengths = mask.sum(dim=1).clamp(min=1)

    # logprob
    gathered = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    gathered = gathered * mask
    seq_logprob = gathered.sum(dim=1) / lengths

    # entropy
    token_entropy = -(probs * log_probs).sum(dim=-1)
    token_entropy = token_entropy * mask
    seq_entropy = token_entropy.sum(dim=1) / lengths

    return seq_logprob.cpu().numpy(), seq_entropy.cpu().numpy()

# -------------------------
def main():
    print("Loading dataset...")
    dataset = load_dataset()
    print(f"Total examples: {len(dataset)}")

    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    # Metrics
    correct = 0
    entropy_choice_correct = 0
    oracle_correct = 0
    hybrid_correct = 0

    correct_entropy = []
    wrong_entropy = []

    all_correct_flags = []
    all_entropies = []

    print("Running evaluation...")

    for ex in tqdm(dataset):
        prompt = build_prompt(ex["question"])

        scores, entropies = score_choices(
            model, tokenizer, prompt, ex["choices"]
        )

        pred = int(np.argmax(scores))
        entropy_pred = int(np.argmin(entropies))
        gold = ex["label"]

        ent_pred_value = entropies[pred]

        # baseline
        if pred == gold:
            correct += 1
            correct_entropy.append(ent_pred_value)
            all_correct_flags.append(1)
        else:
            wrong_entropy.append(ent_pred_value)
            all_correct_flags.append(0)

        all_entropies.append(ent_pred_value)

        # entropy-only
        if entropy_pred == gold:
            entropy_choice_correct += 1

        # oracle
        if pred == gold or entropy_pred == gold:
            oracle_correct += 1

        # -------------------------
        # HYBRID DECISION RULE
        # -------------------------
        threshold = np.mean(entropies)

        if entropies[pred] > threshold:
            hybrid_pred = entropy_pred
        else:
            hybrid_pred = pred

        if hybrid_pred == gold:
            hybrid_correct += 1

    # -------------------------
    # Results
    # -------------------------
    acc = correct / len(dataset)
    entropy_acc = entropy_choice_correct / len(dataset)
    oracle_acc = oracle_correct / len(dataset)
    hybrid_acc = hybrid_correct / len(dataset)

    print("\n=== BASELINE RESULTS ===")
    print(f"Accuracy: {acc:.4f}")

    print("\n=== ENTROPY ANALYSIS ===")
    print(f"Correct mean entropy: {np.mean(correct_entropy):.4f}")
    print(f"Wrong mean entropy:   {np.mean(wrong_entropy):.4f}")

    corr, _ = pearsonr(all_entropies, all_correct_flags)
    print(f"\nEntropy–Correctness Correlation: {corr:.4f}")

    print("\n=== RANKING TEST ===")
    print(f"Entropy-only accuracy: {entropy_acc:.4f}")

    print("\n=== ORACLE TEST ===")
    print(f"Oracle (best of both): {oracle_acc:.4f}")

    print("\n=== HYBRID METHOD ===")
    print(f"Hybrid accuracy: {hybrid_acc:.4f}")

# -------------------------
if __name__ == "__main__":
    main()

Loading dataset...
Total examples: 790
Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running evaluation...


100%|██████████| 790/790 [00:34<00:00, 22.73it/s]


=== BASELINE RESULTS ===
Accuracy: 0.2557

=== ENTROPY ANALYSIS ===
Correct mean entropy: 10.3649
Wrong mean entropy:   10.0650

Entropy–Correctness Correlation: 0.0870

=== RANKING TEST ===
Entropy-only accuracy: 0.2342

=== ORACLE TEST ===
Oracle (best of both): 0.4076

=== HYBRID METHOD ===
Hybrid accuracy: 0.2481


**USING ALPHA AS A PARAMETER**

In [14]:
import json
import requests
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from scipy.stats import pearsonr

# -------------------------
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
def load_dataset():
    url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json"
    data = requests.get(url).json()

    dataset = []
    for item in data:
        mc1 = item["mc1_targets"]
        choices = list(mc1.keys())
        labels = list(mc1.values())

        if len(choices) < 2:
            continue

        correct = labels.index(1)

        dataset.append({
            "question": item["question"],
            "choices": choices,
            "label": correct
        })

    return dataset

# -------------------------
def build_prompt(question):
    return f"Question: {question}\nAnswer:"

# -------------------------
@torch.no_grad()
def score_choices(model, tokenizer, prompt, choices):
    inputs = tokenizer(
        [prompt] * len(choices),
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    targets = tokenizer(
        choices,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    labels = targets.input_ids.clone()
    labels[targets.attention_mask == 0] = -100

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )

    logits = outputs.logits
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()

    target_ids = targets.input_ids
    mask = targets.attention_mask.bool()

    # shift
    log_probs = log_probs[:, :-1, :]
    probs = probs[:, :-1, :]
    target_ids = target_ids[:, 1:]
    mask = mask[:, 1:]

    lengths = mask.sum(dim=1).clamp(min=1)

    # logprob
    gathered = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    gathered = gathered * mask
    seq_logprob = gathered.sum(dim=1) / lengths

    # entropy
    token_entropy = -(probs * log_probs).sum(dim=-1)
    token_entropy = token_entropy * mask
    seq_entropy = token_entropy.sum(dim=1) / lengths

    return seq_logprob.cpu().numpy(), seq_entropy.cpu().numpy()

# -------------------------
def main():
    print("Loading dataset...")
    dataset = load_dataset()
    print(f"Total examples: {len(dataset)}")

    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    # -------------------------
    # Metrics
    # -------------------------
    correct = 0
    entropy_choice_correct = 0
    oracle_correct = 0
    hybrid_correct = 0

    correct_entropy = []
    wrong_entropy = []

    all_correct_flags = []
    all_entropies = []

    print("Running evaluation...")

    # store all scores for alpha search
    all_scores = []
    all_entropies_full = []
    all_labels = []

    for ex in tqdm(dataset):
        prompt = build_prompt(ex["question"])

        scores, entropies = score_choices(
            model, tokenizer, prompt, ex["choices"]
        )

        pred = int(np.argmax(scores))
        entropy_pred = int(np.argmin(entropies))
        gold = ex["label"]

        ent_pred_value = entropies[pred]

        # baseline
        if pred == gold:
            correct += 1
            correct_entropy.append(ent_pred_value)
            all_correct_flags.append(1)
        else:
            wrong_entropy.append(ent_pred_value)
            all_correct_flags.append(0)

        all_entropies.append(ent_pred_value)

        # entropy-only
        if entropy_pred == gold:
            entropy_choice_correct += 1

        # oracle
        if pred == gold or entropy_pred == gold:
            oracle_correct += 1

        # hybrid (kept for record)
        threshold = np.mean(entropies)
        hybrid_pred = entropy_pred if entropies[pred] > threshold else pred
        if hybrid_pred == gold:
            hybrid_correct += 1

        # store for alpha search
        all_scores.append(scores)
        all_entropies_full.append(entropies)
        all_labels.append(gold)

    # -------------------------
    # Base results
    # -------------------------
    acc = correct / len(dataset)
    entropy_acc = entropy_choice_correct / len(dataset)
    oracle_acc = oracle_correct / len(dataset)
    hybrid_acc = hybrid_correct / len(dataset)

    print("\n=== BASELINE RESULTS ===")
    print(f"Accuracy: {acc:.4f}")

    print("\n=== ENTROPY ANALYSIS ===")
    print(f"Correct mean entropy: {np.mean(correct_entropy):.4f}")
    print(f"Wrong mean entropy:   {np.mean(wrong_entropy):.4f}")

    corr, _ = pearsonr(all_entropies, all_correct_flags)
    print(f"\nEntropy–Correctness Correlation: {corr:.4f}")

    print("\n=== RANKING TEST ===")
    print(f"Entropy-only accuracy: {entropy_acc:.4f}")

    print("\n=== ORACLE TEST ===")
    print(f"Oracle (best of both): {oracle_acc:.4f}")

    print("\n=== HYBRID METHOD ===")
    print(f"Hybrid accuracy: {hybrid_acc:.4f}")

    # -------------------------
    # ALPHA SEARCH
    # -------------------------
    alpha_grid = np.linspace(0, 1.0, 11)
    best_alpha = 0
    best_acc = 0
    results = []

    print("\n=== ALPHA SEARCH ===")

    for alpha in alpha_grid:
        correct_alpha = 0

        for scores, entropies, gold in zip(
            all_scores, all_entropies_full, all_labels
        ):
            combined = scores - alpha * entropies
            pred = int(np.argmax(combined))

            if pred == gold:
                correct_alpha += 1

        acc_alpha = correct_alpha / len(dataset)
        results.append((alpha, acc_alpha))

        print(f"alpha={alpha:.2f} → acc={acc_alpha:.4f}")

        if acc_alpha > best_acc:
            best_acc = acc_alpha
            best_alpha = alpha

    print("\n=== BEST RESULT ===")
    print(f"Best alpha: {best_alpha:.2f}")
    print(f"Best accuracy: {best_acc:.4f}")


# -------------------------
if __name__ == "__main__":
    main()

Loading dataset...
Total examples: 790
Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running evaluation...


100%|██████████| 790/790 [00:35<00:00, 22.56it/s]



=== BASELINE RESULTS ===
Accuracy: 0.2557

=== ENTROPY ANALYSIS ===
Correct mean entropy: 10.3649
Wrong mean entropy:   10.0650

Entropy–Correctness Correlation: 0.0870

=== RANKING TEST ===
Entropy-only accuracy: 0.2342

=== ORACLE TEST ===
Oracle (best of both): 0.4076

=== HYBRID METHOD ===
Hybrid accuracy: 0.2481

=== ALPHA SEARCH ===
alpha=0.00 → acc=0.2557
alpha=0.10 → acc=0.2557
alpha=0.20 → acc=0.2582
alpha=0.30 → acc=0.2582
alpha=0.40 → acc=0.2582
alpha=0.50 → acc=0.2595
alpha=0.60 → acc=0.2570
alpha=0.70 → acc=0.2570
alpha=0.80 → acc=0.2557
alpha=0.90 → acc=0.2544
alpha=1.00 → acc=0.2544

=== BEST RESULT ===
Best alpha: 0.50
Best accuracy: 0.2595
